# 09_04 · 손실 함수 오류 분석 (focal 대비)

06_01의 오류 분석 하니스(`src/error_analysis.py`)를 손실 실험에 재사용. **비교 기준 = exp2 focal(len512)**, 대상 = 손실 변형(ZLPR, 이후 ASL·DL2). 로짓은 09_00 방식으로 산출해 `output/`에 둔다.

- **판정 축(loss-function.md)**: k≥2 슬라이스의 **오라클-k 회수율**.
- 임계: `sigmoid τ=0.5 ⟺ logit≥0` = 각 손실 native 임계.
- ASL·DL2는 훈련·로짓 산출 후 config의 `LOSS_MODELS`에 tag를 추가하면 동일 셀이 그대로 처리한다.
- 오류 분석 기법은 모듈의 `TECHNIQUES` 레지스트리로 관리한다 — 기법 추가 = 함수 작성 + 레지스트리 한 줄이면 표·저장에 자동 반영.

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

from huggingface_hub import hf_hub_download
from datasets import load_dataset
from sklearn.metrics import f1_score

# 오류 분석 하니스(계산 로직) — src/error_analysis.py (uv 프로젝트에 editable 설치된 최상위 모듈)
from error_analysis import ErrorAnalysis

In [2]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "split": "test",
    "tau": 0.5,                 # sigmoid τ=0.5 ⟺ logit≥0 = 각 손실 native 임계(focal/zlpr/asl/dl2 공통)
    "raw_ds": "ingyoun/patent-clean-text",
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": str(HF_HOME),
    "out_path": ROOT / "output",
}

# 로짓·라벨 연산만 하므로 tag/arch만 필요(체크포인트·토크나이저는 09_00 참조).
# 비교 기준 = exp2 focal(len512). 대상 = 손실 변형. 판정 축은 손실 vs focal 카디널리티 회수(오라클-k, k≥2).
REF = {"tag": "modernbert-patent-len512", "arch": "modernbert", "kind": "focal"}
LOSS_MODELS = [
    {"tag": "modernbert-patent-len512-zlpr", "arch": "modernbert", "kind": "loss"},
    # ASL·DL2 훈련·로짓 산출(09_00 방식) 후 아래 주석 해제:
    # {"tag": "modernbert-patent-len512-asl", "arch": "modernbert", "kind": "loss"},
    # {"tag": "modernbert-patent-len512-dl2", "arch": "modernbert", "kind": "loss"},
]
MODELS = [REF] + LOSS_MODELS                     # 로드·분석 대상 전체(focal 포함해 나란히 대조)

ANCHOR_TAG = "modernbert-patent-len512-zlpr"     # 앵커 기반 절(계층 판정·cross-Lno·hard-core)이 가리킬 손실 모델

In [3]:
def load_ssot(tag):
    """test SSOT를 스키마 무관하게 정규화(micro/macro/sample/empty_rate)해 반환.
    - 손실 실험: {tag}_test_metrics.json   (flat: test_micro_f1 …, trainer.evaluate 산출)
    - focal/baseline: total_metrics_{tag}.json  (rich: multilabel_f1.keep · empty_rate_tau_micro)
    둘 다 없으면 None.
    """
    flat = config["out_path"] / f"{tag}_test_metrics.json"
    rich = config["out_path"] / f"total_metrics_{tag}.json"
    if flat.exists():
        d = json.loads(flat.read_text(encoding="utf-8"))
        return {"micro": d["test_micro_f1"], "macro": d["test_macro_f1"],
                "sample": d["test_sample_f1"], "empty_rate": d["test_empty_rate"]}
    if rich.exists():
        d = json.loads(rich.read_text(encoding="utf-8"))
        return {**d["multilabel_f1"]["keep"], "empty_rate": d["empty_rate_tau_micro"]}
    return None

In [4]:
random.seed(config['seed'])
np.random.seed(config['seed'])

In [5]:
path = hf_hub_download(
    repo_id="ingyoun/patent-clean-text",
    filename="label_mappings.json",
    repo_type="dataset",
)

with open(path, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

In [6]:
print(type(label_mapping))
print(label_mapping.keys())

<class 'dict'>
dict_keys(['mno2id', 'id2mno', 'mno2lno'])


In [7]:
def show_samples(mappings:dict):
    for i, (k, v) in enumerate(mappings.items()):
        if i <= 5:
            print(f"{k} : {v}")

show_samples(label_mapping["id2mno"])
print("+"*20)
show_samples(label_mapping["mno2lno"])

0 : EA01
1 : EA02
2 : EA03
3 : EA04
4 : EA05
5 : EA06
++++++++++++++++++++
EA01 : EA
EA02 : EA
EA03 : EA
EA04 : EA
EA05 : EA
EA06 : EA


## Hierachy

In [8]:
EA = ErrorAnalysis(label_mapping, num_labels=config["num_labels"], tau=config["tau"])
LS = EA.ls   # 표·저장 셀에서 참조(Lno 축)
print(f"C={LS.C}  L={LS.L}  Lno={LS.lnos}")

C=188  L=17  Lno=['EA', 'EB', 'EC', 'ED', 'EE', 'EF', 'EG', 'EH', 'EI', 'LA', 'LB', 'LC', 'NB', 'NC', 'ND', 'OA', 'OB']


## 데이터 · 라벨

`ingyoun/patent-clean-text`의 `label_ids`로 188차원 다중핫을 만들고, `length_bin`(`kobert_len` 파생, 고정 축)을 슬라이스 축으로 쓴다. 공유 축(정답 `Y`·길이 bin·카디널리티 `k_gold`·`tau`)은 `EvalAxes` 하나로 묶는다.

In [9]:
ds = load_dataset(config["raw_ds"], split=config["split"])
doc_ids = json.loads((config["out_path"] / f"doc_ids_{config['split']}.json").read_text(encoding="utf-8"))

# 로짓 행 순서 == 데이터셋 행 순서 (06_00/09_00이 DataLoader(shuffle=False)로 보장한 축)
assert ds["document_id"] == doc_ids, "로짓 행 순서와 데이터셋 행 순서가 다르다"

EA.set_data(ds)
Y, length_bin, k_gold, BINS = EA.Y, EA.length_bin, EA.k_gold, EA.bins   # 표·verify 셀에서 참조
N = EA.n

print(f"N={N:,} || 라벨 개수 : {np.bincount(k_gold)[1:6].tolist()}(k=1 ~ 5) ||  k>=2 비율 {(k_gold >= 2).mean():.2%}")
print("길이 bin " + "  || ".join(f"{b} : {int((length_bin == b).sum()):,}" for b in BINS))

N=11,271 || 라벨 개수 : [9579, 1278, 298, 67, 33](k=1 ~ 5) ||  k>=2 비율 15.01%
길이 bin <=512 : 3,197  || 512-1024 : 5,183  || 1024-2048 : 2,342  || >2048 : 549


## Logit

In [10]:
EA.add(MODELS, cache_dir=config["out_path"], split=config["split"])   # 로짓 로드→빌드→기법 레지스트리 분석
results, records = EA.models, EA.records
print(list(results))

[load] logits_modernbert-patent-len512_test.npy


[load] logits_modernbert-patent-len512-zlpr_test.npy


['modernbert-patent-len512', 'modernbert-patent-len512-zlpr']


## 앵커(top-1) 오류 분해 — sibling vs cross-Lno

-  sibling 비율이 높다는 것은 모델이 "어느 대분류인지는 대체로 맞히지만, 그 안에서 세부 중분류를 못 가른다"는 뜻이고, 이는 조건부 소프트맥스 $`p(l∣x)⋅p(m∣l,x)`$ 처럼 대분류를 먼저 확정하고 그 안에서 중분류를 고르는 계층 구조가 실제로 도움이 될 여지가 있다는 신호다. 반대로 cross 비율이 높다면 오류가 대분류 경계 자체에서 발생하므로, 계층 구조를 도입해도 1단계(Lno 분류)에서부터 틀려 개선 효과가 제한적이라고 추정할 수 있다.

- sibling 비율은 비정답 클래스에서 균등 추출할 때의 sibling 확률(귀무 기준)과 대비해 읽는다. 계층 확장 여부의 판정은 이 비율이 아니라 「Lno 수준 지표」의 2단계 추정으로 내린다.

In [11]:
anchor = {tag: r["anchor_error"] for tag, r in records.items()}

header = (f"{'tag':<34}{'P@1':>8}{'오류':>8}"
          f"{'sibling':>14}{'cross-Lno':>16}{'우연':>6}{'배수':>8}")
print(header)
print("-" * len(header))

for tag, r in anchor.items():
    sib = f"{r['sibling']:,} ({r['sibling_ratio']:.1%})"
    cro = f"{r['cross_lno']:,} ({1 - r['sibling_ratio']:.1%})"
    print(f"{tag:<34}{r['p@1']:>8.4f}{r['n_error']:>8,}"
          f"{sib:>16}{cro:>16}"
          f"{r['chance_sibling_ratio']:>8.1%}{r['sibling_enrichment']:>7.1f}x")

tag                                    P@1      오류       sibling       cross-Lno    우연      배수
----------------------------------------------------------------------------------------------
modernbert-patent-len512            0.8999   1,128     391 (34.7%)     737 (65.3%)    7.0%    5.0x
modernbert-patent-len512-zlpr       0.8912   1,226     424 (34.6%)     802 (65.4%)    6.8%    5.1x


## 멀티라벨(τ=0.5) 오류 분해 — FP · FN

In [12]:
multilabel = {tag: r["multilabel_error"] for tag, r in records.items()}
print(f"{'tag':<34}{'FP':>8}{'FP sib':>16}{'FN':>8}{'FN sib':>16}{'empty':>11}")

for tag, r in multilabel.items():
    print(f"{tag:<34}{r['fp']:>8,}{r['fp_sibling']:>9,} ({r['fp_sibling_ratio']:>5.1%})"
          f"{r['fn']:>8,}{r['fn_sibling']:>9,} ({r['fn_sibling_ratio']:>5.1%}){r['empty_rate']:>9.2%}")

tag                                     FP          FP sib      FN          FN sib      empty
modernbert-patent-len512             1,916      760 (39.7%)   1,884      661 (35.1%)    1.79%
modernbert-patent-len512-zlpr        2,013      787 (39.1%)   2,066      788 (38.1%)    0.96%


In [13]:
# empty rate가 기존 SSOT와 일치 (focal=rich / 손실=flat 스키마 자동 판별)
ok = 0
for d in MODELS:
    s = load_ssot(d["tag"])
    if s is None:
        print(f"  [warn] SSOT 없음: {d['tag']}")
        continue
    assert abs(multilabel[d["tag"]]["empty_rate"] - s["empty_rate"]) < 1e-4, d["tag"]
    ok += 1
print(f"\nverify: empty rate == SSOT ({ok}/{len(MODELS)} 일치)")


verify: empty rate == SSOT (2/2 일치)


## Lno 수준 지표 · 계층 확장 판정 · 17×17 혼동 행렬

- **판정 기준**: 계층 구조가 이득인지 판정한다. `Mno` 예측을 정답 `Lno` 열로 제한한 **오라클-Lno P@1**(완벽한 Lno 단계를 가정한 상한)에 실제 **Lno 단계 정확도**를 곱한 2단계 추정을 flat P@1과 비교한다.

- cross-Lno 누수가 특정 대분류 쌍에 집중되면 라벨 경계 혼동으로 볼 수 있지만, 균등하면 분류 과제 자체의 구조적 난이도로 해석할 수 있다.

In [14]:
lno = {tag: r["lno_metrics"] for tag, r in records.items()}
confusion = {tag: m.confusion for tag, m in results.items()}

print("[Lno 수준 지표]")
print(f"{'tag':<34}{'micro':>9}{'macro':>9}{'sample':>9}{'P@1':>9}")
for tag, r in lno.items():
    print(f"{tag:<34}{r['micro_f1']:>9.4f}{r['macro_f1']:>9.4f}{r['sample_f1']:>9.4f}{r['p@1']:>9.4f}")

[Lno 수준 지표]
tag                                   micro    macro   sample      P@1
modernbert-patent-len512             0.9068   0.9086   0.9114   0.9346
modernbert-patent-len512-zlpr        0.9022   0.9035   0.9110   0.9288


In [15]:
print("\n[계층 확장 추정 이득]")
print(f"{'tag':<34}{'flat P@1':>10}{'Lno 정확도':>12}{'오라클':>11}{'2단계 추정':>10}{'Δ':>9}")
for d in MODELS:
    tag, r = d["tag"], lno[d["tag"]]
    print(f"{tag:<34}{anchor[tag]['p@1']:>10.4f}{r['p@1']:>14.4f}"
          f"{r['oracle_lno_p@1']:>13.4f}{r['two_stage_p@1_est']:>12.4f}{r['delta_vs_flat']:>+10.4f}")


[계층 확장 추정 이득]
tag                                 flat P@1     Lno 정확도        오라클    2단계 추정        Δ
modernbert-patent-len512              0.8999        0.9346       0.9529      0.8906   -0.0093
modernbert-patent-len512-zlpr         0.8912        0.9288       0.9493      0.8817   -0.0095


In [16]:
hierarchy = EA.hierarchy_verdict(ANCHOR_TAG)
delta = hierarchy["delta_vs_flat"]
print(f"\n판정(기준 {ANCHOR_TAG}): 2단계 추정 {hierarchy['two_stage_p@1_est']:.4f} vs "
      f"flat {hierarchy['flat_p@1']:.4f} ({delta:+.4f}) → {hierarchy['decision']}")


판정(기준 modernbert-patent-len512-zlpr): 2단계 추정 0.8817 vs flat 0.8912 (-0.0095) → flat 유지


In [17]:
M = confusion[ANCHOR_TAG]
off = [(M[g, p], LS.lnos[g], LS.lnos[p]) for g in range(LS.L) for p in range(LS.L) if g != p]
off.sort(reverse=True)
print(f"\n[{ANCHOR_TAG}] cross-Lno 누수 상위 10쌍 (정답 → 예측)")
for cnt, g, p in off[:10]:
    print(f"  {g} → {p}   {cnt:>4,}")
print(f"  off-diagonal 합계 {sum(x[0] for x in off):,} · 상위 10쌍 점유율 "
      f"{sum(x[0] for x in off[:10]) / max(sum(x[0] for x in off), 1):.1%}")


[modernbert-patent-len512-zlpr] cross-Lno 누수 상위 10쌍 (정답 → 예측)
  LC → LB     28
  EA → EI     28
  EA → EE     23
  EI → EA     22
  ND → EH     21
  EA → ND     21
  LA → LB     18
  ND → EA     16
  LB → LC     15
  EH → ND     15
  off-diagonal 합계 896 · 상위 10쌍 점유율 23.1%


## cross-Lno 대칭 쌍 · 국소 처리 상한

In [18]:
pair_analysis = {tag: r["pair_analysis"] for tag, r in records.items()}

for tag, r in pair_analysis.items():
    print(tag)
    print(f"  off-diag {r['off_diagonal_total']:,} · 무향 상위5 {r['top5_share']:.1%} · 상위10 {r['top10_share']:.1%}")
    print(f"  {'쌍':<12}{'a→b':>7}{'b→a':>7}{'합':>6}{'대칭도':>9}")
    for p in r["top5_pairs"]:
        print(f"  {p['pair']:<12}{p['ab']:>7}{p['ba']:>7}{p['total']:>6}{p['symmetry']:>9.3f}")
    g = r["oracle_gain"]
    print(f"  국소 처리 상한(P@1 이득): 상위1 {g['top1']['p@1_gain_pt']:+.2f}pt · "
          f"상위5 {g['top5']['p@1_gain_pt']:+.2f}pt · 상위10 {g['top10']['p@1_gain_pt']:+.2f}pt\n")

modernbert-patent-len512
  off-diag 829 · 무향 상위5 22.2% · 상위10 37.1%
  쌍               a→b    b→a     합      대칭도
  EA<->EI          24     23    47    0.958
  LB<->LC          20     20    40    1.000
  EA<->ND          14     19    33    0.737
  EH<->ND          18     15    33    0.833
  EH<->EI          18     13    31    0.722
  국소 처리 상한(P@1 이득): 상위1 +0.35pt · 상위5 +1.23pt · 상위10 +1.97pt

modernbert-patent-len512-zlpr
  off-diag 896 · 무향 상위5 22.1% · 상위10 35.4%
  쌍               a→b    b→a     합      대칭도
  EA<->EI          28     22    50    0.786
  LB<->LC          15     28    43    0.536
  EA<->ND          21     16    37    0.762
  EH<->ND          15     21    36    0.714
  EA<->EE          23      9    32    0.391
  국소 처리 상한(P@1 이득): 상위1 +0.35pt · 상위5 +1.29pt · 상위10 +2.04pt



In [19]:
# verify — 무향 쌍 질량 합 == off-diagonal · 쌍 상한 ⊆ 전체 오라클-Lno 이득
for d in MODELS:
    tag = d["tag"]
    pairs, off_total = EA.pair_symmetry(tag)
    assert sum(p["total"] for p in pairs) == off_total, tag                    # 무향 쌍이 off-diagonal을 완전 분해
    full_gain = round(100 * (lno[tag]["oracle_lno_p@1"] - anchor[tag]["p@1"]), 3)   # 전체 오라클-Lno 이득(pt)
    top10_gain = pair_analysis[tag]["oracle_gain"]["top10"]["p@1_gain_pt"]
    assert top10_gain <= full_gain + 1e-9, (tag, top10_gain, full_gain)        # 쌍 상한은 전체의 부분집합
print("verify(대칭 쌍) pass — 무향 분해·부분집합 관계 성립")

verify(대칭 쌍) pass — 무향 분해·부분집합 관계 성립


## 라벨 개수 bin — 단일(k=1) vs 다라벨(k≥2)

`modernbert-comparison.md` label-cardinality

In [20]:
count_bin = {tag: r["label_count_bins"] for tag, r in records.items()}

print(f"{'tag':<34}{'bin':>4}{'n':>8}{'micro':>9}{'sample':>9}{'R-Prec':>9}{'FP':>8}{'FN':>8}{'FP:FN':>8}")
for tag, r in count_bin.items():
    for name in ["k=1", "k>=2"]:
        v = r[name]
        print(f"{tag:<34}{name:>4}{v['n']:>8,}{v['micro_f1']:>9.4f}{v['sample_f1']:>9.4f}"
              f"{v['r_precision']:>9.4f}{v['fp']:>8,}{v['fn']:>8,}{v['fp_fn_ratio']:>8.2f}")
    gap = r["k=1"]["sample_f1"] - r["k>=2"]["sample_f1"]
    print(f"{'':<34}{'[sample f1(k=1) - sample f1(k>=2) : ':>7}{'':>8}{'':>9}{gap:>9.4f}]\n")

tag                                bin       n    micro   sample   R-Prec      FP      FN   FP:FN
modernbert-patent-len512           k=1   9,579   0.8822   0.8877   0.8944   1,549     797    1.94
modernbert-patent-len512          k>=2   1,692   0.7994   0.7829   0.8620     367   1,087    0.34
                                  [sample f1(k=1) - sample f1(k>=2) :                     0.1048]

modernbert-patent-len512-zlpr      k=1   9,579   0.8761   0.8855   0.8860   1,646     829    1.99
modernbert-patent-len512-zlpr     k>=2   1,692   0.7741   0.7570   0.8492     367   1,237    0.30
                                  [sample f1(k=1) - sample f1(k>=2) :                     0.1285]



- 멀티 라벨의 성능이 모든 모델에서 저하. 딘일 라벨 샘플이 다수라 모델의 전체 성능이 높게 유지됨. 멀티 라벨의 성능이 단일 라벨에 가려져 있음

## 라벨 개수 bin — 카디널리티 헤드룸 (주 지표 환산)

위 bin 분해는 sample-F1·R-Precision 기준이다. 주 지표는 micro-F1이므로 k≥2의 무게를 **문서 비율이 아니라 양성 라벨 인스턴스 비율**로 환산하고, 결손이 표현(랭킹)에서 오는지 결정 규칙(카디널리티)에서 오는지 가른다 — k≥2 문서에만 **오라클 카디널리티**(정답 개수 `k`를 알고 상위 `k`개 선택, 랭킹 불변)를 적용한 micro-F1이 도달 불가 상한이다.

In [21]:
cardinality = {tag: r["cardinality"] for tag, r in records.items()}

print(f"양성 라벨 인스턴스 {int(Y.sum()):,} · k≥2 점유율 {cardinality[MODELS[0]['tag']]['pos_share_k>=2']:.1%}\n")
print(f"{'tag':<34}{'micro':>9}{'+오라클k':>11}{'이득':>9}{'k≥2과소예측':>13}{'예측/정답':>12}")
for tag, r in cardinality.items():
    kp = f"{r['mean_k_pred_k>=2']:.2f}/{r['mean_k_gold_k>=2']:.2f}"
    print(f"{tag:<34}{r['micro']:>9.4f}{r['micro_oracle_k_on_multi']:>12.4f}"
          f"{r['oracle_k_gain_pt']:>+12.2f}{r['under_predict_rate_k>=2']:>15.1%}{kp:>16}")

양성 라벨 인스턴스 13,564 · k≥2 점유율 29.4%

tag                                   micro      +오라클k       이득      k≥2과소예측       예측/정답
modernbert-patent-len512             0.8601      0.8760       +1.59          44.9%       1.93/2.35
modernbert-patent-len512-zlpr        0.8493      0.8683       +1.89          50.5%       1.84/2.35


In [22]:
# verify — micro 재계산 == SSOT · 오라클-k sample-F1 == R-Precision(k≥2)
for d in MODELS:
    tag = d["tag"]
    s = load_ssot(tag)
    if s is not None:
        assert abs(cardinality[tag]["micro"] - round(s["micro"], 4)) < 1e-4, (tag, cardinality[tag]["micro"], s["micro"])
    else:
        print(f"  [warn] SSOT 없음(micro 대조 생략): {tag}")
    k = Y.sum(1); m2 = k >= 2; logits = results[tag].logits
    cf = np.zeros((int(m2.sum()), Y.shape[1]), dtype=bool)          # k개 정확히 선택 → sample-F1 = R-Precision
    for j, i in enumerate(np.where(m2)[0]):
        cf[j, np.argpartition(-logits[i], k[i])[:k[i]]] = True
    sf1 = round(float(f1_score(Y[m2], cf, average="samples", zero_division=0)), 4)
    assert abs(sf1 - count_bin[tag]["k>=2"]["r_precision"]) < 1e-4, (tag, sf1, count_bin[tag]["k>=2"]["r_precision"])
print("verify(카디널리티) pass — micro SSOT 일치 · 오라클-k sample-F1 == R-Precision(k≥2)")

verify(카디널리티) pass — micro SSOT 일치 · 오라클-k sample-F1 == R-Precision(k≥2)


## 길이 bin × 오류 유형

In [23]:
bin_error = {tag: r["length_bin_error"] for tag, r in records.items()}

for tag, r in bin_error.items():
    print(tag)
    print(f"  {'bin':<12}{'n':>7}{'오류율':>8}{'sibling비':>10}{'FP/문서':>9}{'FN/문서':>8}")
    for b in BINS:
        v = r[b]
        print(f"  {b:<12}{v['n']:>7,}{v['anchor_error_rate']:>9.2%}{v['sibling_ratio']:>11.1%}"
              f"{v['fp_per_doc']:>10.3f}{v['fn_per_doc']:>10.3f}")
    print()

modernbert-patent-len512
  bin               n     오류율  sibling비    FP/문서   FN/문서
  <=512         3,197    8.38%      34.7%     0.154     0.149
  512-1024      5,183    9.65%      33.4%     0.165     0.161
  1024-2048     2,342   11.78%      35.1%     0.190     0.196
  >2048           549   15.30%      40.5%     0.220     0.209

modernbert-patent-len512-zlpr
  bin               n     오류율  sibling비    FP/문서   FN/문서
  <=512         3,197    9.51%      34.5%     0.163     0.158
  512-1024      5,183   10.38%      34.4%     0.175     0.179
  1024-2048     2,342   12.55%      33.0%     0.197     0.216
  >2048           549   16.39%      41.1%     0.228     0.233



## 오류 차집합 — focal → 손실 (손실 성분)

focal(exp2, 동일 창·모델·레시피)을 base로, 각 손실을 target으로 두어 **손실 교체가 top-1 앵커 오류를 어디서 고치고(fixed) 어디서 깨는지(broken)** 본다. fixed 중 **k≥2 문서 비중**이 손실이 겨냥한 카디널리티 표적을 실제로 회수했는지의 top-1 관점 방증이다(주 판정은 위 「카디널리티 헤드룸」 절의 오라클-k 회수율). `hard_core`는 focal·손실이 공통으로 틀린 문서다.

In [24]:
# base=focal(exp2) → target=각 손실. 손실 교체가 top-1 오류를 고친/깬 문서를 가른다.
PAIRS = [(REF["tag"], m["tag"], "loss") for m in LOSS_MODELS]

hc = EA.hard_core()                                        # focal·손실 공통 오류
cross_model = {
    "pairs": [EA.compare(base, target, component) for base, target, component in PAIRS],
    "hard_core": {
        "n": int(hc.sum()),
        "by_length_bin": {b: int((hc & (length_bin == b)).sum()) for b in BINS},
    },
}

for r in cross_model["pairs"]:
    print(f"[{r['component']}] {r['base']} → {r['target']}")
    print(f"  {'':<10}{'n':>7}{'k>=2':>7}{'sibling':>10}{'cross':>8}   " + "".join(f"{b:>12}" for b in BINS))
    for name in ["fixed", "broken"]:
        v = r[name]
        print(f"  {name:<10}{v['n']:>7,}{v['k>=2']:>7,}{v['sibling']:>10,}{v['cross_lno']:>8,}   "
              + "".join(f"{v['by_length_bin'][b]:>12,}" for b in BINS))
    print(f"  {'순이득':<8}{r['net_gain']:>+7,}{'':>25}   "
          + "".join(f"{r['net_by_bin'][b]:>+12,}" for b in BINS))
    print(f"  {'교정률':<8}{'':>7}{'':>25}   "
          + "".join(f"{r['fix_rate_by_bin'][b]:>12.1%}" for b in BINS) + "\n")

print(f"공통 오류(hard core, focal∩손실) {cross_model['hard_core']['n']:,}건")
print("  bin별 " + "  ".join(f"{b} {cross_model['hard_core']['by_length_bin'][b]:,}" for b in BINS))

[loss] modernbert-patent-len512 → modernbert-patent-len512-zlpr


                  n   k>=2   sibling   cross          <=512    512-1024   1024-2048       >2048
  fixed         362     47       138     224             79         180          77          26
  broken        460     67       182     278            115         218          95          32
  순이득         -98                                     -36         -38         -18          -6
  교정률                                               29.5%       36.0%       27.9%       30.9%

공통 오류(hard core, focal∩손실) 766건
  bin별 <=512 189  512-1024 320  1024-2048 199  >2048 58


## 저장

In [25]:
meta = {
    "split": config["split"],
    "n_docs": int(N),
    "tau": config["tau"],
    "num_labels": config["num_labels"],
    "num_lno": LS.L,
    "fields": config["fields"],
    "raw_ds": config["raw_ds"],
    "ref_tag": REF["tag"],
}

# 손실 모델만 저장(focal은 06_01에서 이미 error_analysis_*.json 산출됨 — 덮어쓰지 않는다).
# result = meta + tag/arch + records[tag](기법 레지스트리 결과) — 기법 추가 시 저장에 자동 포함.
for d in LOSS_MODELS:
    tag = d["tag"]
    result = {**meta, "tag": tag, "arch": d["arch"], **records[tag]}
    fp = config["out_path"] / f"error_analysis_{tag}.json"
    fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved: {fp}")

fp = config["out_path"] / "error_analysis_loss_vs_focal.json"
fp.write_text(json.dumps({**meta, **cross_model, "hierarchy_verdict": hierarchy},
                         ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved: {fp}")

saved: C:\workspace\patent_disc\output\error_analysis_modernbert-patent-len512-zlpr.json


saved: C:\workspace\patent_disc\output\error_analysis_loss_vs_focal.json
